# STRATUM — Data Quality Audit

## Phase 2: Exploratory Data Analysis and Preprocessing

### Dataset
Infineon AlCu Physical Vapor Deposition (PVD) Dataset — Version 2

### Objective
Evaluate the integrity, completeness, and consistency of the semiconductor manufacturing dataset before developing virtual metrology models.

### Audit Tasks
1. Verify dataset structure and column names.
2. Identify duplicate wafer records.
3. Analyse missing values.
4. Detect constant and low-variability features.
5. Validate thickness and resistance measurements.
6. Examine quality-label consistency.
7. Summarise preprocessing requirements.

**Note:** This notebook performs read-only analysis. The original dataset will remain unchanged.

In [1]:
# STRATUM - Phase 2.1
# Import libraries and load original dataset

from pathlib import Path
import pandas as pd
import numpy as np

filename = "apc_spc_AlCu_inputs_outputs.csv"

current_dir = Path.cwd()

csv_path = current_dir / "data" / "raw" / filename

if not csv_path.exists():
    csv_path = current_dir.parent / "data" / "raw" / filename

if not csv_path.exists():
    raise FileNotFoundError(
        f"Dataset not found. Current directory: {current_dir}"
    )

df = pd.read_csv(
    csv_path,
    sep=";",
    skiprows=1,
    low_memory=False
)

print("STRATUM - Data Quality Audit")
print("Dataset loaded successfully!")
print("Shape:", df.shape)

STRATUM - Data Quality Audit
Dataset loaded successfully!
Shape: (5208, 178)


## 1. Duplicate Record Analysis

Identify exact duplicate rows and repeated wafer identifiers.

In [2]:
#Duplicate analysis

exact_duplicates = df.duplicated().sum()

duplicate_wafer_ids = df["wafer_id"].duplicated().sum()

print("Total records:", len(df))
print("Exact duplicate rows:", exact_duplicates)
print("Repeated wafer IDs:", duplicate_wafer_ids)

print("\nUnique wafer IDs:", df["wafer_id"].nunique())
print("Unique lots:", df["lot"].nunique())


Total records: 5208
Exact duplicate rows: 0
Repeated wafer IDs: 0

Unique wafer IDs: 5208
Unique lots: 2951


## 2. Missing Value Analysis

Analyse missing-value counts and percentages, and categorise features by missingness severity.

In [3]:
#Missing value analysis

missing_count = df.isna().sum()

missing_percentage = (
    df.isna().mean() * 100
)

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

missing_report = missing_report.sort_values(
    by="Missing Percentage",
    ascending=False
)

# Categorise missingness severity
def missing_category(pct):
    if pct == 0:
        return "No missing values"
    elif pct <= 5:
        return "Low"
    elif pct <= 20:
        return "Moderate"
    elif pct <= 50:
        return "High"
    else:
        return "Very High"

missing_report["Severity"] = (
    missing_report["Missing Percentage"]
    .apply(missing_category)
)

print("Columns with missing values:",
      (missing_count > 0).sum())

print("Columns above 50% missing:",
      (missing_percentage > 50).sum())

display(missing_report.head(20).round(2))

Columns with missing values: 47
Columns above 50% missing: 11


,Missing Count,Missing Percentage,Severity
WAFERCOUNT_RECIPE_MR,3571,68.57,Very High
VOLTS_TARGET_DEP1_G,3534,67.86,Very High
GAS2_DEP1_S,3382,64.94,Very High
WAFERCOUNT_MR,3205,61.54,Very High
LIFE_TARGET_RUN_I,3183,61.12,Very High
T_RUN_ENG,2837,54.47,Very High
PRESS_CHAMBER_DEP1_R,2822,54.19,Very High
PRESS_TRANSFER_RUN_X,2822,54.19,Very High
PRESS_IONGAUGE_BP_I_ENG,2729,52.40,Very High
PRESS_IONGAUGE_BP_I_ENG1,2633,50.56,Very High


## 3. Constant and Low-Variability Features

Identify columns with no variation and examine the variability of numerical process measurements.

In [4]:
#Constant feature analysis

unique_counts = df.nunique(dropna=True)

constant_cols = unique_counts[
    unique_counts <= 1
].index.tolist()

print("Number of constant/all-missing columns:",
      len(constant_cols))

print("\nConstant/all-missing columns:")
for col in constant_cols:
    print(col)

# Inspect cardinality of numerical columns
numeric_cols = df.select_dtypes(
    include=np.number
).columns

numeric_unique_counts = (
    df[numeric_cols]
    .nunique(dropna=True)
    .sort_values()
)

print("\nNumerical columns with fewest distinct values:")
display(numeric_unique_counts.head(20))


Number of constant/all-missing columns: 0

Constant/all-missing columns:

Numerical columns with fewest distinct values:


WAFERCOUNT_RECIPE_MR            2
PROCESS_STATE_RUN_X             2
quality_overall                 2
stable_state                    2
slot                           10
TEMP_CRYOSECONDSTAGE_RUN_X     12
TEMP_CRYOFIRSTSTAGE_RUN_X      23
POWER_TARGET_DEP1_R_ENG        64
TIME_PREV_DIFF_I               97
TEMP_PLATEN_DEP1_X            124
VOLTS_TARGET_DEP1_R           128
T_RUN_PRE2                    146
T_RUN_PRE_PROCESS             153
T_RECIPE                      161
T_RUN_PRE1                    161
RAW_VALUE_9                   172
RAW_VALUE_10                  173
RAW_VALUE_5                   174
RAW_VALUE_4                   175
RAW_VALUE_16                  175
dtype: int64

## 4. Physical Measurement Validation

Check the completeness and numerical validity of thickness, sheet resistance, and derived resistivity measurements.

In [5]:
#Physical measurement validity

thickness_cols = [
    f"RAW_VALUE_t_{i}" for i in range(1, 18)
]

resistance_cols = [
    f"RAW_VALUE_r_{i}" for i in range(1, 18)
]

resistivity_cols = [
    f"RAW_VALUE_{i}" for i in range(1, 18)
]

measurement_groups = {
    "Thickness": thickness_cols,
    "Sheet Resistance": resistance_cols,
    "Derived Resistivity": resistivity_cols
}

results = []

for group_name, columns in measurement_groups.items():

    values = df[columns]

    results.append({
        "Property": group_name,
        "Missing Values": int(values.isna().sum().sum()),
        "Zero Values": int((values == 0).sum().sum()),
        "Negative Values": int((values < 0).sum().sum()),
        "Minimum": values.min().min(),
        "Maximum": values.max().max()
    })

measurement_validation = pd.DataFrame(results)

display(measurement_validation)


,Property,Missing Values,Zero Values,Negative Values,Minimum,Maximum
0,Thickness,0,0,0,133.963108,169.499970
1,Sheet Resistance,0,0,0,0.002568,0.003401
2,Derived Resistivity,0,0,0,0.627148,0.723773


## 5. Quality Label Consistency

Verify that the overall quality label agrees with the three individual material-property quality indicators.

In [6]:
#Quality label consistency

property_quality_cols = [
    "quality_thickness",
    "quality_resistance",
    "quality_resistivity"
]

# Standardise labels only for the consistency check
quality_values = (
    df[property_quality_cols]
    .astype("string")
    .apply(lambda col: col.str.strip().str.lower())
)

valid_property_labels = (
    quality_values.isin(["good", "bad"])
    .all(axis=1)
)

valid_overall_labels = (
    df["quality_overall"].isin([0, 1])
)

valid_rows = (
    valid_property_labels & valid_overall_labels
)

derived_overall = (
    quality_values.eq("bad")
    .any(axis=1)
    .astype(int)
)

inconsistent_rows = (
    valid_rows &
    derived_overall.ne(df["quality_overall"])
)

print("Rows with invalid/missing quality labels:",
      (~valid_rows).sum())

print("Rows with inconsistent overall quality:",
      inconsistent_rows.sum())

print("\nRecorded overall quality:")
print(df["quality_overall"].value_counts())


Rows with invalid/missing quality labels: 0
Rows with inconsistent overall quality: 0

Recorded overall quality:
quality_overall
0    4873
1     335
Name: count, dtype: int64


In [7]:
# Dataset audit summary

audit_summary = pd.DataFrame({
    "Check": [
        "Total records",
        "Total columns",
        "Exact duplicate rows",
        "Repeated wafer IDs",
        "Columns with missing values",
        "Columns above 50% missing",
        "Constant/all-missing columns",
        "Invalid quality-label rows",
        "Inconsistent overall-quality rows"
    ],
    "Result": [
        len(df),
        len(df.columns),
        int(exact_duplicates),
        int(duplicate_wafer_ids),
        int((missing_count > 0).sum()),
        int((missing_percentage > 50).sum()),
        len(constant_cols),
        int((~valid_rows).sum()),
        int(inconsistent_rows.sum())
    ]
})

display(audit_summary)


,Check,Result
0,Total records,5208
1,Total columns,178
2,Exact duplicate rows,0
3,Repeated wafer IDs,0
4,Columns with missing values,47
5,Columns above 50% missing,11
6,Constant/all-missing columns,0
7,Invalid quality-label rows,0
8,Inconsistent overall-quality rows,0
